# Component Evaluation Notebook

This notebook automatically finds the project folder in Google Drive by searching for project marker files. It works both when the project is in your `MyDrive` and when it is available through a shared folder shortcut.

If the shared project folder is not found, add it to your Drive as a shortcut and run the first cell again.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive")


def find_project_dir() -> Path:
    marker_files = [
        "colab_utils/requirements_colab.txt",
        "colab_utils/colab_launcher.py",
        "app/chatbot.py",
    ]

    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for requirements_file in root.rglob("requirements_colab.txt"):
            candidate = requirements_file.parents[1]

            if all((candidate / marker).exists() for marker in marker_files):
                candidates.append(candidate)

    candidates = sorted(set(candidates), key=lambda path: str(path).lower())

    if not candidates:
        raise FileNotFoundError(
            "Project folder not found. If the folder was shared with you, "
            "open Google Drive, go to 'Shared with me', right-click the project folder, "
            "choose 'Add shortcut to Drive', then run this cell again."
        )

    if len(candidates) == 1:
        return candidates[0]

    print("Multiple project folders found:")
    for index, candidate in enumerate(candidates, start=1):
        print(f"{index}. {candidate}")

    choice = input("Choose the project folder number, or press Enter to use the first one: ").strip()
    if not choice:
        return candidates[0]

    return candidates[int(choice) - 1]


PROJECT_DIR = find_project_dir()
COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"
REQUIREMENTS_PATH = COLAB_UTILS_DIR / "requirements_colab.txt"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())
print("Requirements file:", REQUIREMENTS_PATH)

In [ ]:
print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH" 

In [ ]:
import os
import sys
import importlib
import logging

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("HF_TOKEN not found. Public models can still be loaded, but loading may fail.")

os.environ["APP_DEBUG"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s:%(message)s",
    force=True,
)

print("Project configured.")

In [ ]:
%ls evaluation/

In [ ]:
!python -m evaluation.eval_router --model qwen3 --batch-size 4

In [ ]:
!python -m evaluation.eval_NLU --model qwen3 --batch-size 4

In [ ]:
!python -m evaluation.eval_DM --model qwen3 --batch-size 4

In [ ]:
!python -m evaluation.eval_NLG --model qwen3 --batch-size 4

In [ ]:
from google.colab import files

files.download("evaluation/results/nlu_results.json")
files.download("evaluation/results/nlu_errors.json")